In [0]:
from pyspark.sql import functions as F

## Brand

In [0]:
df_brand_br = spark.table('ecommerce.bronze.bronze_brands')
df_brand_br.show(10)

In [0]:
df_brand_sil = df_brand_br.withColumn('brand_name',F.trim('brand_name'))
df_brand_sil.show(10) 

In [0]:
df_brand_sil = df_brand_sil.withColumn('brand_code',F.regexp_replace('brand_code',r'[^A-Za-z0-9]',''))
df_brand_sil.show(10)

In [0]:
df_brand_sil.select('category_code').distinct().show()

In [0]:
repetitions = {
    'BOOKS':'BKS',
    'GROCERY':'GRCY',
    'TOYS':'TOY'
}

In [0]:
df_brand_sil = df_brand_sil.replace(repetitions,subset='category_code')

In [0]:
df_brand_sil.select('category_code').distinct().show()

In [0]:
df_brand_sil.select('brand_code').distinct().show()

In [0]:
df_brand_sil.select('brand_name').distinct().show()

In [0]:
df_brand_sil.write.format('delta').mode('overwrite').option('mergeSchema',True).saveAsTable('ecommerce.silver.silver_brands')

## Category

In [0]:
df_category_br = spark.table("ecommerce.bronze.bronze_category")
df_category_br.display()

In [0]:
df_dups = df_category_br.groupBy('category_code').count().filter("count > 1").show()

In [0]:
df_category_sil = df_category_br.dropDuplicates(['category_code'])

In [0]:
df_category_sil = df_category_sil.withColumn('category_code',F.upper(F.col('category_code')))

In [0]:
df_category_sil.display()

In [0]:
df_category_sil.write.format('delta').mode("overwrite").option('mergeSchema',True).saveAsTable('ecommerce.silver.silver_category')

## Products

In [0]:
df_products_br = spark.read.table('ecommerce.bronze.bronze_products')
df_products_br.display()

In [0]:
all_chars = df_products_br.select(F.explode(F.split(F.col('weight_grams'),'')).alias('chars')).distinct().orderBy('chars')

In [0]:
all_chars.display()

In [0]:
df_products_sil = df_products_br.withColumn('weight_grams',F.regexp_replace(F.col('weight_grams'),'g','').cast('double'))
df_products_sil.select('weight_grams').display()
df_products_sil.printSchema()

In [0]:
df_products_sil = df_products_sil.withColumn('length_cm',F.regexp_replace(F.col('length_cm'),',','.').cast('double'))
df_products_sil.printSchema()

In [0]:
df_products_sil = df_products_sil.withColumn('category_code',F.upper('category_code')).withColumn('brand_code',F.upper('brand_code'))
df_products_sil.select('category_code','brand_code').show(1)

In [0]:
df_products_sil.display()

In [0]:
df_products_sil.select('material').distinct().show()

In [0]:
df_products_sil = (df_products_sil
                   .withColumn('material',
                               F.when(F.col('material')=='Coton','Cotton')
                               .when(F.col('material')=='Alumium','Aluminium')
                               .when(F.col('material')=='Ruber','Rubber')
                               .otherwise(F.col('material'))))
df_products_sil.select('material').distinct().show()                               

In [0]:
df_products_sil = df_products_sil.withColumn('rating_count',F.when(F.col('rating_count').isNotNull(),F.abs(F.col('rating_count')))
                                             .otherwise(F.lit(0)))

In [0]:
df_products_sil.select('rating_count').distinct().show()

In [0]:
df_products_sil.select('color').distinct().show()

In [0]:
c_nulls = df_products_sil.filter(F.col('color').isNull()).count()
c_nulls

In [0]:
df_products_sil = df_products_sil.fillna({'color':'Unknown'})

In [0]:
c_nulls = df_products_sil.filter(F.col('color')=='Unknown').count()
c_nulls

In [0]:
df_products_sil.write.format('delta').option('schemaMerge',True).mode('overwrite').saveAsTable('ecommerce.silver.silver_products')

## Customers

In [0]:
df_customers_br = spark.read.table('ecommerce.bronze.bronze_customer')

In [0]:
df_customers_br.display()

In [0]:
c_nulls = df_customers_br.filter(F.col('customer_id').isNull()).count()
c_nulls

In [0]:
df_customers_sil = df_customers_br.filter(F.col('customer_id').isNotNull())

In [0]:
c_nulls = df_customers_br.filter(F.col('phone').isNull()).count()
c_nulls

In [0]:
df_customers_sil = df_customers_sil.fillna({'phone':'Not Available'})

In [0]:
c_nulls = df_customers_sil.filter(F.col('phone').isNull()).count()
c_nulls

In [0]:
df_customers_sil.select('country_code').distinct().show()
df_customers_sil.select('country').distinct().show()

In [0]:
df_customers_sil.write.format('delta').option('schemaMerge',True).mode('overwrite').saveAsTable('ecommerce.silver.silver_customers')

## Date

In [0]:
df_date_br = spark.read.table('ecommerce.bronze.bronze_date')
df_date_br.display()

In [0]:
df_date_sil = df_date_br.withColumn('date',F.to_date(F.col('date'),'dd-MM-yyyy'))

In [0]:
df_date_sil.printSchema()

In [0]:
d_dups  = df_date_sil.groupBy('date').count().filter('count>1')
d_dups.display()

In [0]:
df_date_sil = df_date_sil.dropDuplicates(['date'])

In [0]:
df_date_sil = df_date_sil.withColumn('day_name',F.initcap(F.col('day_name')))

df_date_sil.display()

In [0]:
df_date_sil = df_date_sil.withColumn('week_of_year',F.abs(F.col('week_of_year')).cast('int').cast('string'))
df_date_sil.show()

In [0]:
df_date_sil =df_date_sil.withColumn('quarter',F.concat(F.lit('Q'),F.col('quarter'),F.lit('-'),F.col('year')))
df_date_sil =df_date_sil.withColumn('week_of_year',F.concat(F.lit('Week'),F.col('week_of_year'),F.lit('-'),F.col('year')))
df_date_sil.display()

In [0]:
df_date_sil.withColumnRenamed('week_of_year','week')

In [0]:
df_date_sil.write.format('delta').option('mergeSchema',True).mode('overwrite').saveAsTable('ecommerce.silver.silver_date')

## Orders

In [0]:
df_orders_br = spark.read.table('ecommerce.bronze.bronze_orders')

In [0]:
df_orders_br.display()

In [0]:
c_nulls = [F.count(F.when(F.col(c).isNull(),c)).alias(c) for c in df_orders_br.columns]
df_orders_br.select(c_nulls).display()

In [0]:
df_orders_sil =df_orders_br.withColumn('unit_price',F.replace(F.col('unit_price'),F.lit('$'),F.lit('')).cast('double'))

In [0]:
df_orders_sil.printSchema()

In [0]:
df_orders_sil.select('quantity').distinct().show()

In [0]:
df_orders_sil = df_orders_sil.withColumn('quantity',F.when(F.col('quantity')=='Two','2').otherwise(F.col('quantity')))

In [0]:
df_orders_sil.select('quantity').distinct().show()

In [0]:
df_orders_sil.select('discount_pct').distinct().show()
df_orders_sil.select('channel').distinct().show()


In [0]:
df_orders_sil.write.format('delta').option('mergeSchema',True).mode('overwrite').saveAsTable('ecommerce.silver.silver_orders')